# Instructor Notebook 05 — Encoders & Sentence Embeddings
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. Requires sentence-transformers (already in conda env).
> First run downloads `all-MiniLM-L6-v2` (~80MB) — do this before class.

**Learning arc:**
1. Encoder vs Decoder — two paths from one architecture
2. Sentence embeddings — dense vectors that capture meaning
3. Closing the semantic gap: 'court order' ≈ 'judicial mandate'
4. Adversarial examples — where embeddings still fail
5. HIPAA classification with sentence embeddings
6. Building intuition for where LLM extraction fits in

In [ ]:
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', '-q'],
               capture_output=True)

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print('Loading sentence transformer model (all-MiniLM-L6-v2)...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}')
print('This model runs on CPU — no GPU needed.')

---
## Part 1 · Encoder vs Decoder

> **Say:** "The Transformer architecture splits into two families based on what you want the model to do."

| | **Encoder (BERT)** | **Decoder (GPT, Gemma)** |
|---|---|---|
| **Direction** | Bidirectional (sees left AND right) | Left-to-right only |
| **Training task** | Predict masked words | Predict next word |
| **Use for** | Understanding, classification, extraction | Generation, completion |
| **Our example** | Sentence embeddings | LLM oracle extraction |

> **Say:** "Today we use an encoder (MiniLM). It reads a full sentence bidirectionally and outputs a single 384-dimensional vector that represents the sentence's meaning. No generation — pure understanding."

---
## Part 2 · Closing the Semantic Gap

> **Say:** "Remember TF-IDF's fatal flaw: 'court order' and 'judicial mandate' similarity = 0.08. Same legal concept, zero word overlap. Sentence embeddings should fix this."

> **Say:** "Let's measure."

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Four groups of phrases
# Group A: court orders (PERMITTED under HIPAA 164.512(e))
# Group B: law enforcement with no order (DENIED)
# Group C: adversarial — hedged/implied court orders
# Group D: treatment context (PERMITTED for different reason)

groups = {
    'A — Court Order (PERMITTED)': [
        "A court order was issued requiring release of the records",
        "A judicial mandate compelled disclosure of patient information",
        "The subpoena issued by the court authorized disclosure",
    ],
    'B — No Legal Process (DENIED)': [
        "The officer verbally requested the patient records",
        "Law enforcement asked the hospital to turn over the file",
        "Police said they needed the records for their investigation",
    ],
    'C — Adversarial / Hedged (DENIED)': [
        "The officer implied there might be a court order forthcoming",
        "Officials suggested they had judicial authorization",
        "Acting pursuant to what might be a legal mandate",
    ],
    'D — Treatment Context (PERMITTED)': [
        "The physician requested records for treatment purposes",
        "The hospital disclosed records to the treating specialist",
        "Records shared with the consulting physician for ongoing care",
    ],
}

all_phrases = [p for phrases in groups.values() for p in phrases]
short_labels = (
    ['A1','A2','A3'] + ['B1','B2','B3'] + ['C1','C2','C3'] + ['D1','D2','D3']
)

# TF-IDF similarity
tfidf = TfidfVectorizer(ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(all_phrases)
sim_tfidf = cosine_similarity(X_tfidf)

# Sentence embedding similarity
embs = model.encode(all_phrases, show_progress_bar=False)
sim_emb = cosine_similarity(embs)

print('Cosine similarity: court order (A1) vs all other phrases')
print()
print(f"  {'Label':<5} {'TF-IDF':>8} {'Embedding':>11}  {'Verdict':>10}  Phrase")
print('  ' + '-' * 85)
for i, (label, phrase) in enumerate(zip(short_labels, all_phrases)):
    tfidf_s = sim_tfidf[0][i]
    emb_s   = sim_emb[0][i]
    verdict = 'PERMITTED' if label in ['A1','A2','A3','D1','D2','D3'] else 'DENIED'
    diff    = '↑' if emb_s > tfidf_s + 0.2 else ('↓' if emb_s < tfidf_s - 0.1 else ' ')
    print(f"  {label:<5} {tfidf_s:>8.3f} {emb_s:>11.3f} {diff}  {verdict:>10}  {phrase[:45]}")

print()
print(f'KEY: A1 (court order) ↔ A2 (judicial mandate):')
print(f'  TF-IDF similarity:   {sim_tfidf[0][1]:.3f}  (nearly 0 — different words)')
print(f'  Embedding similarity: {sim_emb[0][1]:.3f}  (high — same meaning)')
print(f'  Gap closed: +{sim_emb[0][1] - sim_tfidf[0][1]:.3f}')

In [ ]:
# The adversarial problem — group C is close to A in embedding space
print('ADVERSARIAL ANALYSIS:')
print('How similar are hedged phrases (C) to real court orders (A)?')
print()

a_indices = [0, 1, 2]
c_indices = [6, 7, 8]
b_indices = [3, 4, 5]

print('Embedding similarity between adversarial (C) and court orders (A):')
print(f"  {'C phrase':>40}  {'avg sim to A':>14}")
print('  ' + '-' * 60)
for ci in c_indices:
    avg_sim_to_A = np.mean([sim_emb[ci][ai] for ai in a_indices])
    avg_sim_to_B = np.mean([sim_emb[ci][bi] for bi in b_indices])
    print(f"  {all_phrases[ci][:40]:<40}  {avg_sim_to_A:>14.3f}")

print()
print('Embedding similarity between denied (B) and court orders (A):')
for bi in b_indices:
    avg_sim_to_A = np.mean([sim_emb[bi][ai] for ai in a_indices])
    print(f"  {all_phrases[bi][:40]:<40}  {avg_sim_to_A:>14.3f}")

print()
print('→ Adversarial C phrases are CLOSE to A (court orders) in embedding space.')
print('   They talk ABOUT court orders. They are not court orders.')
print('   Embeddings capture topic, not legal validity.')
print('   This is why LLM extraction still makes ~6% of errors.')

---
## Part 3 · HIPAA Classification with Embeddings

> **Say:** "Let's apply this to all 137 HIPAA scenarios. Use sentence embeddings as features instead of TF-IDF, and see if the accuracy improves."

In [ ]:
HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'
hipaa = pd.read_csv(HIPAA_PATH)
texts  = hipaa['question'].fillna('').tolist()
y      = (hipaa['ground_truth'] == 'PERMITTED').astype(int).values

print('Encoding 137 HIPAA scenarios with sentence transformer...')
print('(~2-5 seconds on CPU)')
X_emb = model.encode(texts, batch_size=32, show_progress_bar=True)
print(f'\nEmbedding matrix shape: {X_emb.shape}')
print(f'  → 137 scenarios × 384 dimensions')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF for comparison
tfidf_full = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))
X_tfidf = tfidf_full.fit_transform(texts)

# Classifiers
lr = LogisticRegression(max_iter=2000, random_state=42)

scores_tfidf = cross_val_score(lr, X_tfidf, y, cv=5, scoring='accuracy')
scores_emb   = cross_val_score(lr, X_emb, y, cv=5, scoring='accuracy')

print('HIPAA classification accuracy comparison:')
print()

comparison = [
    ('Majority class baseline',           max(y.mean(), 1-y.mean()),   '—'),
    ('TF-IDF (unigrams+bigrams) + LogReg', scores_tfidf.mean(),        f'±{scores_tfidf.std():.3f}'),
    ('Sentence embeddings + LogReg',       scores_emb.mean(),          f'±{scores_emb.std():.3f}'),
    ('LLM extraction + formal engine',     0.9416,                     '(Gemma3-4B, 129/137)'),
]

for method, acc, note in comparison:
    bar = '█' * int(acc * 40)
    print(f'  {method:<42} {acc:>7.1%}  {note}')
    print(f'  {"":42} {bar}')
    print()

print(f'Embedding improvement over TF-IDF: +{scores_emb.mean()-scores_tfidf.mean():.1%}')
print(f'LLM+engine advantage over embeddings: +{0.9416-scores_emb.mean():.1%}')

---
## Part 4 · Semantic Search — Find Similar Cases

> **Say:** "Embeddings let us do something impossible with TF-IDF: semantic search. Find cases that are LEGALLY SIMILAR even if they use different words."

> **Say:** "This is directly relevant to your research: if you write an adversarial scenario, you can find the most similar real cases in the benchmark."

In [ ]:
def semantic_search(query, top_k=5):
    """Find most similar HIPAA scenarios to a query using cosine similarity."""
    q_emb = model.encode([query], show_progress_bar=False)
    sims   = cosine_similarity(q_emb, X_emb)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    
    print(f'Query: "{query}"')
    print(f'Top {top_k} most similar HIPAA scenarios (by embedding similarity):')
    print()
    for rank, idx in enumerate(top_indices, 1):
        verdict = hipaa['ground_truth'].iloc[idx]
        match   = hipaa['match'].iloc[idx]
        sim     = sims[idx]
        print(f'  {rank}. Sim={sim:.3f} | {verdict} | {'✓' if match=="Y" else '✗'} | {texts[idx][:120]}...')
    print()

# Search for court order scenarios
semantic_search("judicial mandate requiring disclosure of medical records")

# Search for treatment scenarios
semantic_search("physician disclosed patient information for ongoing care")

In [ ]:
# Show the 8 LLM failure cases and find what their nearest neighbors predicted
failures = hipaa[hipaa['match'] == 'N'].reset_index(drop=True)
print(f'LLM failure analysis: {len(failures)} cases the model got wrong')
print()

for _, row in failures.iterrows():
    idx = hipaa[hipaa['row_id'] == row['row_id']].index[0]
    q_emb = X_emb[idx:idx+1]
    sims  = cosine_similarity(q_emb, X_emb)[0]
    sims[idx] = -1  # exclude self
    nn_idx = sims.argmax()
    nn_verdict = hipaa['ground_truth'].iloc[nn_idx]
    nn_sim     = sims[nn_idx]
    
    error_type = 'FN (missed PERMITTED)' if row['ground_truth']=='PERMITTED' else 'FP (wrong PERMITTED)'
    print(f'  Case {row["row_id"]:>4} | {error_type}')
    print(f'           Truth={row["ground_truth"]} Pred={row["verdict_norm"]}')
    print(f'           Text: {str(row["question"])[:90]}...')
    print(f'           Nearest neighbor (sim={nn_sim:.3f}): {nn_verdict} case')
    print()

---
## Summary

| Concept | Takeaway |
|---|---|
| **Encoder (BERT-style)** | Reads sentence bidirectionally → single dense vector |
| **Sentence embeddings** | Geometric meaning: similar phrases → nearby vectors |
| **Closed the gap** | 'court order' ↔ 'judicial mandate': TF-IDF=0.08 → Embedding=0.85+ |
| **Still fails** | Adversarial hedging: 'implied court order' embeds near real court order |
| **Why LLMs win** | They read the FULL sentence, attend to hedging words like 'implied', 'might be' |

**The progression so far:**
- Rules → ML (FIFA) → DL (non-linearity) → NLP (text as numbers) → Encoders (meaning)
- Each step solved the previous step's problem. Each left a new gap.
- **Next: LLMs** — close the remaining gap with generation-based extraction.